# Week 4 — DiffusionGemma（DLM）微調＋評測 on Colab Pro
**這份 notebook 只負責 Week 4 的 Colab 部分（DLM 那一半）**；AR 那一半與 MoE 主線都在本機跑，
對應 `week4_執行手冊.md` Step 5。手冊講「為什麼」，這裡是「怎麼跑」。

## 開跑前的額度決策（手冊 §額度預算）
右上角「資源檢視器」→ 查剩餘 compute units：
- **≥ 40 units**：完整跑（微調 + base/tuned 各一輪快速評測，A100 約 3 小時）
- **20–40 units**：跑微調 + 快速評測，跳過完整評測
- **< 20 units**：只跑 §5–6 的 **base 快速評測**（DLM tuned 缺席，報告註明「額度不足，降級為 baseline-only 對比」——這是事前講好的降級路徑，不是失敗）

## 一致性規則（沿用前三週）
1. 訓練資料與 Week 2/3 同一個抽樣（seed 42、8,000 筆）。
2. LoRA scaling = **2.0**（`lora_alpha=32, r=16`）——Week 3 已證明 scaling 不是崩潰主因，但 2.0 仍是常規值。
3. **200 步**：Week 3 checkpoint 掃描顯示 400 步後格式崩加速。
4. CoT 用 **drop**（不含 think 內文）+ **5% \box 格式樣本**（科目與評測三科零重疊）。
5. 評測固定種子、strict/lenient、macro/micro 並列；**DLM base vs tuned 必須同 session 同 config**。
6. Colab 的吞吐/記憶體數字**不可**和本機 M4 Pro 的數字並列（跨硬體不可比）。

---
# §1 環境

In [ ]:
#@title 1.1 GPU 檢查（DLM 微調/評測需要 ~52GB bf16 → A100 80GB / H100）
import subprocess, os, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
# 官方 notebook 明寫：128 個 MoE 專家是 3-D 融合張量，bnb 4bit 壓不了（~46GB 恆為 bf16）。
# 所以 40GB A100 會 offload 到「慢到不可用」——微調與評測都一樣，別硬跑。
FORCE_RUN = False  #@param {type:"boolean"}
EVAL_ONLY = False   # （已停用：DLM base 品質改在本機用 scripts/dlm_cli_eval.py 跑）
if VRAM_GB < 70 and not FORCE_RUN:
    raise SystemExit(
        f"❌ 這台 GPU 只有 {VRAM_GB:.0f}GB，DiffusionGemma 需要 ~52GB bf16。\n"
        "   執行階段 → 變更執行階段類型 → A100 80GB / H100。\n"
        "   沒有大卡可選就走降級路徑：base 品質本機 dlm_cli_eval.py 已涵蓋，\n"
        "   tuned 記為「硬體不可得，降級」（報告注明）。明知會 offload 仍要試 → 勾 FORCE_RUN。")
print(f"GPU: {GPU}  VRAM: {VRAM_GB:.0f} GB  bf16: {torch.cuda.is_bf16_supported()}")
print("⚠️ 80GB 卡燒 units 快（H100 更快），跑完一個 stage 就存檔，不要掛機閒置。")

In [ ]:
#@title 1.2 安裝套件（比照官方 DiffusionGemma notebook 的版本釘選）
%%capture
!pip install unsloth
!pip install --no-deps --upgrade --force-reinstall "unsloth_zoo>=2026.6.5" "unsloth>=2026.6.5"
!pip install --no-deps transformers==5.11.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0" "datasets==4.3.0" peft accelerate pandas pyarrow

In [ ]:
#@title 1.2b 驗收：每一項都要能 import（unsloth 只驗安裝、不 import）
import importlib, importlib.util, importlib.metadata
for name in ["transformers","datasets","peft","accelerate","pandas"]:
    m = importlib.import_module(name)
    print(f"  {name:<14} {getattr(m,'__version__','?')}")
assert importlib.util.find_spec("unsloth"), "unsloth 沒裝到"
print(f"  unsloth        {importlib.metadata.version('unsloth')}（不在這格 import）")
import transformers
assert hasattr(transformers, "DiffusionGemmaForBlockDiffusion"), \
    "transformers 沒有 DiffusionGemma 類別 —— 版本不對，重跑 1.2 後重啟工作階段"
print("DiffusionGemma 類別 ✅")

In [ ]:
#@title 1.3 掛 Drive、路徑、續跑狀態（斷線後先重跑這格）
import os, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ultrascale_week4'); ROOT.mkdir(parents=True, exist_ok=True)
for d in ["data","out","results","reports"]: (ROOT/d).mkdir(exist_ok=True)

def done_path(tag): return ROOT/'results'/f'{tag}.json'
def is_done(tag):   return done_path(tag).exists()
def mark_done(tag, obj): done_path(tag).write_text(json.dumps(obj, ensure_ascii=False, indent=2))

print("狀態：")
for t in ["data_dlm","train_dlm","eval_dlm_base_quick","eval_dlm_tuned_quick",
          "eval_dlm_base_full","eval_dlm_tuned_full"]:
    print(f"  {'✅' if is_done(t) else '⬜'} {t}")

In [ ]:
#@title 1.4 HuggingFace 登入（DiffusionGemma 是 Apache 2.0，不用接受授權，但登入下載較快）
from huggingface_hub import login
import os
if os.environ.get("HF_TOKEN"): login(os.environ["HF_TOKEN"])
else:
    try: login()
    except Exception as e: print("略過登入：", e)
# ⚠️ xet 在 Colab 上會抓出不完整 snapshot（W2 本機也踩過）→ 一律走傳統下載路徑
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 優先用 unsloth 的 4-bit 鏡像（~18 GB），沒有才退回 google 官方 bf16（~52 GB）
from huggingface_hub import repo_exists, snapshot_download
_CANDIDATES = ["unsloth/diffusiongemma-26B-A4B-it-unsloth-bnb-4bit",
               "unsloth/diffusiongemma-26B-A4B-it-bnb-4bit",
               "unsloth/diffusiongemma-26B-A4B-it",
               "google/diffusiongemma-26B-A4B-it"]
MODEL_DLM = next(r for r in _CANDIDATES if repo_exists(r))
print("模型（解析後）：", MODEL_DLM)

def ensure_model_downloaded(repo=None, tries=5):
    """先把權重抓齊再交給 from_pretrained。snapshot_download 可斷點續傳，
    網路中斷就重試，已下載的檔案不會重來。"""
    repo = repo or MODEL_DLM
    for i in range(tries):
        try:
            snapshot_download(repo, max_workers=4)
            print(f"✅ {repo} 權重齊了")
            return repo
        except Exception as e:
            print(f"  下載中斷 {i+1}/{tries}（{type(e).__name__}: {str(e)[:80]}）→ 續傳重試…")
    raise SystemExit("網路太不穩：換個時段或重開 Runtime 再跑這格（已下載部分保留續傳）")

---
# §2 資料
和 Week 2/3 **同一個抽樣**（tw-reasoning-instruct-50k、seed 42、8,000 筆），
但用 DiffusionGemma 的 chat template 重新渲染：
- CoT 一律 **drop**（DiffusionGemma 有 Gemma 4 式 thinking channel，但 Week 2 的教訓是
  訓練資料的散文 CoT 會洗掉格式遵循；乾脆不餵，回答只留 output）。
- 混入 **5% \box 格式樣本**，科目取自 TMMLU+ **非評測科目**（評測三科零重疊）。
- **資料洩漏檢查是全量、全文、確定性比對**（Week 3 教訓：`set` 順序隨機會「上次過、這次炸」）。

In [ ]:
#@title 2.1 下載、抽樣、渲染、洩漏檢查
import json, random, re
import pandas as pd
from datasets import load_dataset
from pathlib import Path

EVAL_SUBJECTS = ["geography_of_taiwan","taiwanese_hokkien","three_principles_of_people"]
MIX_RATIO, N_SAMPLE, SEED = 0.05, 8000, 42

SYS_BOX = (
    "使用者將提供一個題目，並附上選項 A、B、C、D。\n"
    "請仔細閱讀題目要求，根據題意選出最符合的選項，並將選項以以下格式輸出：\n"
    "\\box{選項}\n"
    "請確保僅將選項包含在 { } 中，否則將不計算為有效答案。\n"
    "務必精確遵循輸出格式，避免任何多餘內容或錯誤格式。\n"
    "例如：答案是 A，就輸出 \\box{A}。\n")

# TMMLU+ 直接用本機 repo 打包好的 parquet（和本機評測是同一份資料）。
# ikala/tmmluplus 的 HF repo 原始檔是 csv，snapshot_download 抓不到 parquet —— 別走那條路。
TMMLU = Path('/content/tmmluplus')
if not any(TMMLU.glob("*.parquet")):
    zp = ROOT/'data'/'tmmluplus_parquet.zip'
    assert zp.exists(), (
        "找不到 TMMLU+ 資料。請把本機 repo 根目錄的 tmmluplus_parquet.zip（3.6 MB）"
        "上傳到 Drive 的 ultrascale_week4/data/ 之後重跑這格。")
    import zipfile, shutil
    with zipfile.ZipFile(zp) as z:
        z.extractall('/content/_tmmlu_unzip')
    TMMLU.mkdir(exist_ok=True)
    for f in Path('/content/_tmmlu_unzip').rglob('*.parquet'):
        shutil.copy(f, TMMLU/f.name)
subjects_all = sorted(x.stem for x in TMMLU.glob("*.parquet"))
assert len(subjects_all) >= 60, f"只解出 {len(subjects_all)} 科，zip 內容不對？"
assert all(any(s in x for x in subjects_all) for s in EVAL_SUBJECTS), "評測三科不在資料裡！"
print(f"TMMLU+ 科目 {len(subjects_all)} ✅（來源：本機 repo 的 parquet，與本機評測同一份）")

if is_done("data_dlm"):
    print("[skip] data_dlm")
else:
    ds = load_dataset("twinkle-ai/tw-reasoning-instruct-50k", split="train")
    rng = random.Random(SEED)
    idx = rng.sample(range(len(ds)), N_SAMPLE)
    rows = []
    for i in idx:
        ex = ds[i]
        # 欄位：conversations(ShareGPT) / think / output —— 只取 user 問句 + output（drop CoT）
        user = None
        for t in ex.get("conversations") or []:
            if t.get("from") in ("human","user"): user = t.get("value"); break
        out = (ex.get("output") or "").strip()
        if user and out:
            rows.append({"messages":[{"role":"user","content":user.strip()},
                                     {"role":"assistant","content":out}]})
    # 先建評測題全集（給洩漏排除用 —— TMMLU+ 有跨科目重複題，Week 3 教訓）
    eval_q_set = set()
    for s in EVAL_SUBJECTS:
        f = next(x for x in TMMLU.glob("*.parquet") if s in x.stem)
        eval_q_set |= {str(q) for q in pd.read_parquet(f)["question"].tolist()}

    # 5% box 格式樣本（非評測科目，且題目層級排除跨科目重複題）
    n_mix = int(len(rows)*MIX_RATIO)
    pool = [s for s in subjects_all if not any(e in s for e in EVAL_SUBJECTS)]
    mix, prng = [], random.Random(SEED)
    while len(mix) < n_mix:
        s = pool[prng.randrange(len(pool))]
        df = pd.read_parquet(TMMLU/f"{s}.parquet")
        r = df.iloc[prng.randrange(len(df))]
        if str(r["question"]) in eval_q_set: continue   # 跨科目重複題 → 生成端就排除
        opts = "\n".join(f"{k}: {r[k]}" for k in "ABCD" if pd.notna(r.get(k)))
        gold = str(r["answer"]).strip().upper()
        if gold not in "ABCD": continue
        mix.append({"messages":[
            {"role":"system","content":SYS_BOX},
            {"role":"user","content": f"{r['question']}\n{opts}"},
            {"role":"assistant","content": f"\\box{{{gold}}}"}]})
    data = rows + mix
    random.Random(SEED).shuffle(data)

    # 洩漏檢查：全量、全文、確定性（排序後逐題）。
    # 過短的題幹（如「何者為非？」）是通用片語，子字串比對必假陽性 ——
    # 這類題不帶資訊量（沒有選項就無意義），且 mix 在生成端已做「題目精確相等」排除，
    # 所以只對 >=10 字的題幹做全文比對。
    blob = "\n".join(m["content"] for d in data for m in d["messages"])
    MIN_LEN = 10
    checked = [q for q in sorted(eval_q_set) if q and len(q) >= MIN_LEN]
    skipped = len(eval_q_set) - len(checked)
    leaks = [q for q in checked if q in blob]
    assert not leaks, f"❌ 資料洩漏 {len(leaks)} 題！例：{leaks[:2]}"
    print(f"洩漏檢查通過（{len(checked)} 題全文比對；{skipped} 題過短通用題幹由生成端精確排除涵蓋）")

    with (ROOT/'data'/'dlm_train.jsonl').open('w') as f:
        for d in data: f.write(json.dumps(d, ensure_ascii=False)+"\n")
    mark_done("data_dlm", {"n": len(data), "n_mix": n_mix, "seed": SEED, "cot": "drop"})
    print(f"→ {len(data)} 筆（含 {n_mix} 筆 box 格式樣本）")

---
# §3–4 微調（A100 80GB / H100）
照官方 `DiffusionGemma_(26B-A4B)-Sudoku.ipynb` 的做法（已核對原始碼）：
- **bf16 全量載入**（`load_in_4bit=False`）——bnb 壓不了 3-D 融合專家，~52GB 是硬需求；
- **不是 SFTTrainer**：block-diffusion 目標 = 把答案塞進 256-token canvas、隨機污染、
  預測乾淨版，CE 只算在答案 token 上；
- LoRA 掛 backbone attention + dense MLP（官方預設），專家凍結；scaling 2.0；
- 官方 500 步 @A100 約 14 分鐘 → 我們 200 步約 6–8 分鐘，訓練本身很便宜，貴在載入與大卡時租。

In [ ]:
#@title 3.1 載入 DiffusionGemma（bf16——官方指定）+ LoRA r16/alpha32
import torch, unsloth
from unsloth import FastModel
Loader = getattr(unsloth, "FastDiffusionModel", FastModel)
print("Loader:", Loader.__name__)
ensure_model_downloaded()
free_gb = torch.cuda.mem_get_info()[0]/1e9
if free_gb < 50:
    print(f"[warn] 可用 {free_gb:.0f}GB < 50GB：權重會 offload，官方警告會慢到不可用")
# ⚠️ load_in_4bit 必須 False：bnb 壓不了 3-D 融合專家（官方 notebook 原話），4bit 省不了那 46GB
model, tokenizer = Loader.from_pretrained(MODEL_DLM, dtype=torch.bfloat16, load_in_4bit=False)
processor = tokenizer                      # diffusion checkpoint 帶的是 processor
tok = processor.tokenizer if hasattr(processor, "tokenizer") else processor
vocab = model.config.text_config.vocab_size
canvas_len = model.config.canvas_length
print("vocab", vocab, "| canvas", canvas_len)
# 官方 get_peft_model 預設 target = backbone attention + dense MLP，128 個融合專家凍結。
# 官方用 r=64/alpha=128（scaling 2.0）；我們 r=16/alpha=32 —— 同 scaling、參數更省，
# 與本機 26B 方案 A 的 r16/scale2.0 口徑一致。
model = Loader.get_peft_model(model, r=16, lora_alpha=32, use_gradient_checkpointing=False)
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"可訓練參數 {n_tr/1e6:.2f}M；載入後 GPU {torch.cuda.memory_allocated()/2**30:.2f} GiB")
assert n_tr > 1e6, "可訓練參數異常少 -- LoRA 沒掛上?"

In [ ]:
#@title 4.1 Block-diffusion 微調 200 步（官方訓練迴圈改編；不是 SFTTrainer）
if is_done("train_dlm"):
    print("[skip] train_dlm")
else:
    import json, time, random, torch
    rows = [json.loads(l) for l in (ROOT/'data'/'dlm_train.jsonl').open()]
    eos = (model.generation_config.eos_token_id or [1])
    eos = eos[0] if isinstance(eos, (list, tuple)) else eos
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos

    def build_examples(rows):
        # 官方 recipe：目標塞進單一 256-token canvas，隨機污染後預測乾淨版；
        # 超過 canvas 的答案略過（保留率要印出來、寫進報告）。
        out, skipped = [], 0
        for r in rows:
            msgs = r["messages"]
            prompt_ids = processor.apply_chat_template(
                msgs[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt")[0]
            ids = tok.encode(msgs[-1]["content"], add_special_tokens=False)
            content = ids + [eos]
            if len(content) > canvas_len:
                skipped += 1; continue
            x0 = torch.tensor(content + [pad]*(canvas_len-len(content)), dtype=torch.long)
            mask = torch.zeros(canvas_len, dtype=torch.bool); mask[:len(content)] = True
            out.append((prompt_ids, x0, mask))
        return out, skipped

    examples, skipped = build_examples(rows)
    keep = len(examples)/(len(examples)+skipped)
    print(f"可用樣本 {len(examples)}（>256 token 略過 {skipped}，保留率 {keep:.0%} —— 報告要註明）")
    assert len(examples) > 500, "可用樣本太少 —— 檢查資料渲染"

    dev = next(p.device for p in model.parameters() if p.device.type != "meta")
    STEPS, GRAD_ACCUM, LR, T_LO = 200, 4, 1e-4, 0.1   # 200 步：W3 checkpoint 曲線的教訓
    random.seed(42); torch.manual_seed(42)
    model.config.use_cache = True; model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LR, betas=(0.9, 0.95), weight_decay=0.0)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS,
                                                pct_start=0.1, anneal_strategy="cos")

    def corrupt(x0):
        t = random.uniform(T_LO, 1.0)
        xt = x0.to(dev).clone()
        m = torch.rand(canvas_len, device=dev) < t
        xt[m] = torch.randint(0, vocab, (canvas_len,), device=dev)[m]
        return xt.unsqueeze(0)

    order = list(range(len(examples))); ptr = 0; t0 = time.time()
    torch.cuda.reset_peak_memory_stats()
    opt.zero_grad(set_to_none=True); losses = []
    for step in range(1, STEPS+1):
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            if ptr >= len(order):
                random.shuffle(order); ptr = 0
            prompt_ids, x0, lm = examples[order[ptr]]; ptr += 1
            out = model(input_ids=prompt_ids.unsqueeze(0).to(dev),
                        canvas_ids=corrupt(x0), self_conditioning_logits=None)
            logits = out.logits[0].float()
            m = lm.to(dev)
            loss = torch.nn.functional.cross_entropy(logits[m], x0.to(dev)[m])
            (loss/GRAD_ACCUM).backward(); step_loss += loss.item()/GRAD_ACCUM
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        losses.append(step_loss)
        if step % 10 == 0:
            print(f"step {step:3d}/{STEPS} | loss {step_loss:.4f} | {time.time()-t0:.0f}s", flush=True)
    assert losses[-1] < losses[0], "loss 沒降 —— 停下來查（別急著評測）"
    stats = {"minutes": (time.time()-t0)/60, "peak_gib": torch.cuda.max_memory_allocated()/2**30,
             "first_loss": losses[0], "final_loss": losses[-1],
             "n_examples": len(examples), "kept_ratio": keep}
    print(stats)
    model.save_pretrained(str(ROOT/'out'/'dlm-lora-200'))
    processor.save_pretrained(str(ROOT/'out'/'dlm-lora-200'))
    # ⚠️ 官方警告：不要 merge（會弄壞 clipped LoRA linears）。評測一律用掛載方式。
    mark_done("train_dlm", stats)

---
# §5 評測器
和 Week 3 §7.1 **同一套計分**（固定種子、strict/lenient、macro/micro），
外加一條 DLM 專屬規則：**計分前剝掉 `<|channel>thought … <channel|>`**，
避免 thought 裡的「答案是 X」被寬鬆解析誤抓（Week 2 嫌疑之一）。
生成用 DiffusionGemma 預設 sampler（Entropy-Bounded、max 48 步、adaptive stopping）——
不要自己改 sampler 參數，**base 和 tuned 必須同一組設定**。

In [ ]:
#@title 5.1 評測函式（固定種子；生成走官方 generation_config 路徑）
import re, json, random, time, copy, torch
import pandas as pd

BOX = [r"\\{1,2}box{([A-Z])}", r"\\{1,2}boxed{([A-Z])}"]
LEN = [r"box\{\s*([ABCD])\s*\}",
       r"(?:答案是|答案為|正確答案是|應該是|選項)\s*[:：]?\s*([ABCD])"]
def strip_thinking(s):
    return re.sub(r"<\|channel>thought.*?(<channel\|>|$)", "", s or "", flags=re.S)
def ex_strict(s):
    for p in BOX:
        m = re.search(p, s or "")
        if m: return m.group(1)
    return None
def ex_lenient(s):
    a = ex_strict(s)
    if a: return a
    for p in LEN:
        m = re.search(p, s or "")
        if m: return m.group(1).upper()
    return None
def shuffle_options(row, rng):
    opts = [(k, row[k]) for k in "ABCD" if k in row and pd.notna(row[k])]
    gold = row.get(str(row['answer']).strip().upper())
    if gold is None: return None
    rng.shuffle(opts)
    new = {"question": row['question']}
    for (old, t), nk in zip(opts, "ABCD"):
        new[nk] = t
        if t == gold: new['answer'] = nk
    return new if 'answer' in new else None
def build_prompt(q):
    return q['question'] + "\n" + "\n".join(f"{k}: {v}" for k, v in q.items()
                                             if k not in ("question","answer"))

DEV = next(p.device for p in model.parameters() if p.device.type != "meta")

def dlm_generate(msgs, max_new_tokens=256):
    ids = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to(DEV)
    gc_ = copy.deepcopy(model.generation_config)
    gc_.max_new_tokens = max_new_tokens          # sampler 其餘用官方預設（EB、48 步、adaptive）
    torch.manual_seed(0)
    with torch.no_grad():
        out = model.generate(input_ids=ids, generation_config=gc_)
    seq = out.sequences[0, ids.shape[1]:] if hasattr(out, "sequences") else out[0][ids.shape[1]:]
    return tok.decode(seq.tolist(), skip_special_tokens=True), len(seq)

@torch.no_grad()
def evaluate_dlm(tag, limit=100, max_new_tokens=256, seed=42):
    if is_done(f"eval_{tag}"):
        print(f"[skip] eval_{tag}"); return json.loads(done_path(f"eval_{tag}").read_text())
    torch.cuda.reset_peak_memory_stats()
    model.eval()
    per, recs, gen_tok, gen_s = {}, [], 0, 0.0
    for s in EVAL_SUBJECTS:
        f = next(x for x in TMMLU.glob("*.parquet") if s in x.stem)
        df = pd.read_parquet(f)
        rng = random.Random(seed)
        qs = [q for q in (shuffle_options(r, rng) for _, r in df.iterrows()) if q]
        if limit: qs = qs[:limit]
        ok_s = ok_l = unp = 0
        for i, q in enumerate(qs):
            msgs = [{"role":"system","content":SYS_BOX},   # 不加 <|think|> → thinking off
                    {"role":"user","content": build_prompt(q)}]
            t0 = time.time()
            txt, n_new = dlm_generate(msgs, max_new_tokens)
            dt = time.time() - t0
            body = strip_thinking(txt)
            ps, pl = ex_strict(body), ex_lenient(body)
            ok_s += ps == q['answer']; ok_l += pl == q['answer']; unp += ps is None
            gen_tok += n_new; gen_s += dt
            recs.append({"subject": s, "gold": q['answer'], "pred_strict": ps,
                         "pred_lenient": pl, "seconds": round(dt,2), "output": txt[:1200]})
            if (i+1) % 20 == 0:
                print(f"    {s} {i+1}/{len(qs)} 嚴格 {ok_s/(i+1):.3f} 無法解析 {unp/(i+1):.3f}", flush=True)
        n = len(qs)
        per[s] = {"n": n, "acc_strict": ok_s/n, "acc_lenient": ok_l/n, "unparsed": unp/n}
        print(f"  {s:<30} n={n} 嚴格 {ok_s/n:.4f} 無法解析 {unp/n:.4f}")
    tot = sum(v['n'] for v in per.values()); k = len(per)
    res = {"tag": tag, "n": tot, "seed": seed, "thinking": "off", "engine": "colab-bf16",
           "macro_acc_strict": sum(v['acc_strict'] for v in per.values())/k,
           "micro_acc_strict": sum(v['acc_strict']*v['n'] for v in per.values())/tot,
           "macro_unparsed":  sum(v['unparsed'] for v in per.values())/k,
           "micro_unparsed":  sum(v['unparsed']*v['n'] for v in per.values())/tot,
           "tok_per_s_colab": gen_tok/gen_s if gen_s else None,
           "peak_gib": torch.cuda.max_memory_allocated()/2**30, "per_subject": per,
           "note": "Colab bf16；吞吐/記憶體不可與本機數字並列"}
    with (ROOT/'results'/f'eval_{tag}.jsonl').open('w') as f:
        for r in recs: f.write(json.dumps(r, ensure_ascii=False)+"\n")
    mark_done(f"eval_{tag}", res)
    return res
print("ok")

In [ ]:
#@title 5.2 防呆：3 題（base 可能不套 \box —— 通順散文可續跑，亂碼要停）
def probe3():
    f = next(x for x in TMMLU.glob("*.parquet") if EVAL_SUBJECTS[0] in x.stem)
    df = pd.read_parquet(f); rng = random.Random(42)
    qs = [q for q in (shuffle_options(r, rng) for _, r in df.head(5).iterrows()) if q][:3]
    ok, coherent = 0, 0
    for q in qs:
        msgs = [{"role":"system","content":SYS_BOX},{"role":"user","content":build_prompt(q)}]
        txt, _ = dlm_generate(msgs)
        got = ex_strict(strip_thinking(txt))
        print(f"  正解 {q['answer']} | 抽到 {got} | {strip_thinking(txt).strip()[:100]!r}")
        ok += got is not None
        coherent += len(strip_thinking(txt).strip()) > 5
    assert coherent > 0, "輸出是空/亂碼 —— 生成管線壞了，停"
    if ok == 0:
        print("  ⚠️ 3 題都沒 \box 但輸出通順 —— DLM base 的格式行為本來就是量測對象，續跑")
    else:
        print(f"防呆通過（{ok}/3 抽得出 \box）")
print("ok")

---
# §6 跑評測
**順序**：base quick → tuned quick →（額度夠才跑）full。
base 和 tuned 都在**同一個 session、同一套 sampler 設定**（官方預設 EB sampler、
max 48 步、adaptive stopping）——engine 標 `colab-bf16`，與本機 llama.cpp 的數字分開列。
tuned 用掛 adapter 的方式載，**不 merge**（官方警告 merge 會弄壞 clipped LoRA linears）。

In [ ]:
#@title 6.1 DLM base：快速評測（每科 100 題；同 engine 的 base，供 tuned 對照）
FULL_EVAL = False  #@param {type:"boolean"}
probe3()
if not is_done("eval_dlm_base_quick"):
    r = evaluate_dlm("dlm_base_quick", limit=100)
    print(r["micro_acc_strict"], r["micro_unparsed"])
if FULL_EVAL and not is_done("eval_dlm_base_full"):
    evaluate_dlm("dlm_base_full", limit=None)

In [ ]:
#@title 6.2 DLM tuned：掛 adapter 後同一套評測（不 merge —— 官方警告）
from peft import PeftModel
adapter_dir = str(ROOT/'out'/'dlm-lora-200')
if not hasattr(model, "peft_config"):        # 斷線重來：base + adapter 掛回
    model = PeftModel.from_pretrained(model, adapter_dir)
n_lora = len([n for n, _ in model.named_modules() if "lora" in n.lower()])
print("adapter 掛載模組數：", n_lora); assert n_lora > 0
probe3()
if not is_done("eval_dlm_tuned_quick"):
    r = evaluate_dlm("dlm_tuned_quick", limit=100)
    print(r["micro_acc_strict"], r["micro_unparsed"])
if FULL_EVAL and not is_done("eval_dlm_tuned_full"):
    evaluate_dlm("dlm_tuned_full", limit=None)

In [ ]:
#@title 7.1 匯總：印出要抄回本機 repo 的結果
import json
print("把 Drive 的 ultrascale_week4/results/ 整個資料夾抓回本機 repo 的 results/week4/colab/\n")
rows = []
for t in ["dlm_base_quick","dlm_tuned_quick","dlm_base_full","dlm_tuned_full"]:
    if is_done(f"eval_{t}"):
        r = json.loads(done_path(f"eval_{t}").read_text())
        rows.append((t, r))
        print(f"{t:<22} micro嚴格 {r['micro_acc_strict']:.4f}  macro嚴格 {r['macro_acc_strict']:.4f}  "
              f"無法解析 {r['micro_unparsed']:.4f}  tok/s(Colab) {r.get('tok_per_s_colab') or 0:.1f}")
if is_done("train_dlm"):
    print("\ntrain:", done_path("train_dlm").read_text())

---
# §8 出狀況怎麼辦
- **OOM / offload 警告**：這個模型在 <50GB 可用 VRAM 上就是不行，換 A100 80GB / H100，
  不要調參硬擠（offload 之後一步要幾分鐘，白燒 units）。
- **loss 不降或恆為 0**：先看 §4.1 的 build_examples 保留率——答案超過 256 token 的樣本
  都被略過，保留率過低代表資料形態不合 canvas 訓練；分佈極端就把 box-mix 比例拉高重生資料。
- **額度中途燒完**：狀態檔都在 Drive，換帳號/等額度後重跑 §1，完成的 stage 會 skip。
- **無法解析率異常高**：看 §6 記錄的 output——分清楚「沒學會格式」vs「thought channel
  沒剝乾淨」（後者是評測 bug，Week 2 差點誤判過）。
- **API 對不上**：以官方 notebook 為準（unslothai/notebooks → DiffusionGemma Sudoku），
  只搬 model/訓練迴圈的寫法，資料與超參數（r16/alpha32/200 步/seed42）維持本 notebook。